In [1]:
import jax.numpy as jnp
import jax.random as jr
from jaxtyping import Array, Scalar

import bayinx as byx
from bayinx import define
from bayinx.dists import Normal, Poisson
from bayinx.flows import LowRankAffine
from bayinx.nodes import Continuous, Observed
from bayinx.ops import exp


# Define model
class PoissonGLM(byx.Model):
    beta: Continuous[Scalar] = define(shape = 'n_predictors')

    X: Observed[Array] = define(shape = ('n_obs', 'n_predictors'))
    y: Observed[Array] = define(shape = 'n_obs')

    def model(self, target):
        # Priors
        self.beta << Normal(0.0, 10.0)

        # Compute expected response
        mu = exp(self.X @ self.beta)

        # Accumulate likelihood
        self.y << Poisson(mu)

        return target

# Simulate example
n_obs = 100
n_predictors = 5
X: Array = jr.normal(jr.key(0), (n_obs, n_predictors - 1))
X = jnp.column_stack((jnp.ones((n_obs,)), X))
beta = jnp.array(range(n_predictors))

y = jr.poisson(jr.key(0), jnp.exp(X @ beta), (n_obs, ))

# Define posterior
posterior = byx.Posterior(PoissonGLM,
    n_obs = n_obs,
    n_predictors = n_predictors,
    X = X,
    y = y
)

# Configure and fit
posterior.configure(flowspecs = [LowRankAffine(2)])
posterior.fit(max_iters = int(1e5), learning_rate = 1e-2)

# Compute posterior mean estimate
pp = posterior.sample('beta', 10000)
mean_est = pp.mean(0)
print(mean_est)


[-0.0406254  1.0151039  2.012866   3.0274272  3.999844 ]


In [6]:
import arviz
import numpy as onp

In [11]:
ppn = onp.array(pp)
## avoid FutureWarning about how 2d arrays are interpreted by arviz.hdi ...
hdi_est = [arviz.hdi(ppn[:,i]) for i in range(ppn.shape[1])]
## could flatten/bind for prettiness?
hdi_est

[array([-0.06568051, -0.0152132 ], dtype=float32),
 array([1.006465 , 1.0239804], dtype=float32),
 array([2.0056312, 2.020359 ], dtype=float32),
 array([3.0110848, 3.0431178], dtype=float32),
 array([3.9919462, 4.0079374], dtype=float32)]

In [12]:
type(pp)

jaxlib._jax.ArrayImpl